Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_trino_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()

In [ ]:
import trino
import pandas as pd


query = """

WITH
date_range AS (
    SELECT
        CASE
            WHEN day_of_week(current_date) = 1
                THEN current_date - interval '2' day
            ELSE current_date - interval '1' day
        END AS start_dt,
        current_date - interval '1' day AS end_dt
),

disposition_form AS (
    SELECT
        ticket_uuid,
        max_by(CASE WHEN q_name = 'ticket_tagging' THEN answer END, CASE WHEN q_name = 'ticket_tagging' THEN answer_id END) AS form_ticket_tagging,
        max_by(CASE WHEN q_name = 'qrfc'            THEN answer END, CASE WHEN q_name = 'qrfc'            THEN answer_id END) AS form_qrfc,
        max_by(CASE WHEN q_name = 'ticket_type'     THEN answer END, CASE WHEN q_name = 'ticket_type'     THEN answer_id END) AS form_ticket_type,
        max_by(CASE WHEN q_name = 'product'         THEN answer END, CASE WHEN q_name = 'product'         THEN answer_id END) AS disposition_product
    FROM (
        SELECT
            da.id AS answer_id,
            da.ticket_uuid,
            q.name AS q_name,
            coalesce(o.text, da.answer_text) AS answer
        FROM crm_tickets.disposition_answers da
        CROSS JOIN date_range dr
        INNER JOIN crm_tickets.questions q
            ON da.question_reference_id = q.reference_id
            AND q.__deleted = 'false'
            AND q.name IN ('qrfc', 'ticket_type', 'ticket_tagging', 'product')
        LEFT JOIN crm_tickets.options o
            ON json_extract_scalar(da.option_reference_ids, '$[0]') = o.reference_id
            AND o.__deleted = 'false'
        WHERE da.__deleted = 'false'
          AND da.is_active = true
          AND da.created_at >= cast(date_add('day', -7, dr.start_dt) AS varchar)
          AND da.created_at < cast(date_add('day', 60, dr.end_dt) AS varchar)

        UNION ALL

        SELECT
            da.id AS answer_id,
            da.ticket_uuid,
            q.name AS q_name,
            coalesce(o.text, da.answer_text) AS answer
        FROM crm_ntl_tickets.disposition_answers da
        CROSS JOIN date_range dr
        INNER JOIN crm_ntl_tickets.questions q
            ON da.question_reference_id = q.reference_id
            AND q.__deleted = 'false'
            AND q.name IN ('qrfc', 'ticket_type', 'ticket_tagging', 'product')
        LEFT JOIN crm_ntl_tickets.options o
            ON json_extract_scalar(da.option_reference_ids, '$[0]') = o.reference_id
            AND o.__deleted = 'false'
        WHERE da.__deleted = 'false'
          AND da.is_active = true
          AND da.created_at >= cast(date_add('day', -7, dr.start_dt) AS varchar)
          AND da.created_at < cast(date_add('day', 60, dr.end_dt) AS varchar)
    ) x
    GROUP BY 1
),

prompts AS (
    SELECT ticket_id, prompt, created_at FROM corp_analytics.krisha_chat_ct_tbl
    UNION
    SELECT ticket_id, prompt, created_at FROM corp_analytics.trisha_ct_chat_tbl
    UNION
    SELECT ticket_id, prompt, created_at FROM corp_analytics.esha_ct_tbl
),

prompts1 AS (
    SELECT
        ticket_id,
        max_by(
            trim(regexp_extract(prompt, 'Customer query categories List:\s*\[([^\]]*)\]', 1)),
            created_at
        ) AS last_prompt,
        min_by(
            trim(regexp_extract(prompt, 'Customer query categories List:\s*\[([^\]]*)\]', 1)),
            created_at
        ) AS first_prompt
    FROM prompts
    WHERE length(trim(regexp_extract(prompt, 'Customer query categories List:\s*\[([^\]]*)\]', 1))) > 0
    GROUP BY ticket_id
),

base AS (
    SELECT
        a.*,
        GREATEST(
            COALESCE(
                date_diff(
                    'second',
                    CAST(a.first_agent_assigned_time_obcs AS timestamp),
                    CAST(a.resolved_time AS timestamp)
                ),
                0
            ),
            0
        ) AS aht_seconds
    FROM corp_analytics.cx_sub_ticket_data_model_tbl a
    CROSS JOIN date_range dr
    WHERE
        CAST(substring(a.ticket_created_time, 1, 10) AS date) BETWEEN dr.start_dt AND dr.end_dt
        AND a.source_channel = 'CHAT'
        AND a.overall_last_agent_type_for_sub_ticket = 'OUTBOUND_CALL_SUPPORT_AGENT_L1'
        AND a.bot_description IN (
            'Inbound_Call_Mercury_GI',
            'Inbound_Call_Mercury_HL',
            'Inbound_Call_Mercury_PL',
            'Inbound_Call_on_Mercury'
        )
        AND (
            CASE
                WHEN a.overall_last_agent_type_for_sub_ticket LIKE '%OUTBOUND%'
                    THEN a.obcs_l1_agent_sub_disposition
            END
        ) NOT IN (
            'Blank Call',
            'Call disconnected by customer while assisting the customer',
            'RNR',
            'Short call'
        )
        AND (
            CASE
                WHEN a.overall_last_agent_type_for_sub_ticket LIKE '%OUTBOUND%'
                    THEN a.obcs_l1_agent_sub_disposition
            END
        ) IS NOT NULL
        AND (
            CASE
                WHEN a.overall_last_agent_type_for_sub_ticket LIKE '%OUTBOUND%'
                    THEN a.obcs_l1_agent_sub_disposition
            END
        ) <> ''
)

SELECT DISTINCT
    'INBOUND' AS call_type,
    CAST(substring(a.ticket_created_time, 1, 10) AS date) AS req_date,
    a.canonical_ticket_id,
    a.csat,
    a.ticket_group AS title,
    a.first_obcs_l1_agent_name AS email_id,
    a.phone_number AS pl_before_the_ticket,
    CASE
        WHEN a.overall_last_agent_type_for_sub_ticket LIKE '%OUTBOUND%' THEN a.obcs_l1_agent_disposition
        WHEN a.overall_last_agent_type_for_sub_ticket = 'SUPPORT_AGENT_L1' THEN a.l1_agent_disposition
        WHEN a.overall_last_agent_type_for_sub_ticket = 'SUPPORT_AGENT_L2' THEN a.l2_agent_disposition
    END AS agent_filled_reason,
    CASE
        WHEN a.overall_last_agent_type_for_sub_ticket LIKE '%OUTBOUND%' THEN a.obcs_l1_agent_sub_disposition
        WHEN a.overall_last_agent_type_for_sub_ticket = 'SUPPORT_AGENT_L1' THEN a.l1_agent_sub_disposition
        WHEN a.overall_last_agent_type_for_sub_ticket = 'SUPPORT_AGENT_L2' THEN a.l2_agent_sub_disposition
    END AS agent_filled_sub_reason,
    dr.start_dt AS extracted_date1,
    dr.end_dt AS extracted_date2,
    CAST(a.aht_seconds AS double) / 60.0 AS AHT_in_minutes,
    a.aht_seconds AS AHT_in_sec,

    coalesce(
        df.form_ticket_tagging,
        nullif(trim(a.latest_qrfc), ''),
        df.form_qrfc,
        df.form_ticket_type,
        b.ticket_tag,
        xz.ticket_tag
    ) AS qrfc,

    df.disposition_product

FROM base a
CROSS JOIN date_range dr
LEFT JOIN crm_tickets.tickets xz
    ON a.ticket_id = xz.id
   AND a.canonical_ticket_id = xz.canonical_ticket_id
   AND xz.agent_type <> 'SYSTEM'
   AND xz.created_at >= cast(date_add('day', -1, dr.start_dt) AS varchar)
   AND xz.created_at < cast(date_add('day', 2, dr.end_dt) AS varchar)
LEFT JOIN crm_ntl_tickets.tickets b
    ON a.ticket_id = b.id
   AND a.canonical_ticket_id = b.canonical_ticket_id
   AND b.agent_type <> 'SYSTEM'
   AND b.created_at >= cast(date_add('day', -1, dr.start_dt) AS varchar)
   AND b.created_at < cast(date_add('day', 2, dr.end_dt) AS varchar)
LEFT JOIN disposition_form df
    ON df.ticket_uuid = cast(coalesce(xz.uuid, b.uuid) AS varchar)
LEFT JOIN prompts1 p1
    ON a.ticket_id = p1.ticket_id
"""

df = get_trino_df(query)
if not df.empty:
    display(df)

In [ ]:
# Python cell
import os
import shutil

if not os.path.exists("data"):
    os.makedirs("data")

gsheet_url = "https://docs.google.com/spreadsheets/d/1mV_Dczgu_Qx1yolj8c8u-SO3K06HnOnjBVpx5XZO-ZY/edit?gid=1102365521#gid=1102365521"
worksheet_name = "Master Dump"
gs_reader = GoogleSheetReader(gsheet_url, service_account)

gs_reader.clear_sheets(
    worksheet_name,
    start_row=1,
    start_column=1
)

combined_df = pd.concat([df], ignore_index=True) 

gs_reader.set_df_in_sheets( 
worksheet_name, 
combined_df, 
include_headers=True, 
row=1, 
col=1
)